# Sketch2Shape AI — Colab GPU Worker Runner

This notebook sets up the isolated Python 3.10 environments for **ControlNet** and **TripoSR** on a Google Colab T4 GPU, launches the **Worker API Service (`worker.py`)**, and exposes it through a secure Cloudflare Tunnel.

### Prerequisites:
1. Select **Runtime -> Change runtime type -> T4 GPU**.
2. Run the cells sequentially from top to bottom.
3. Copy the generated **Public Worker Tunnel URL** and your token into your local `backend/.env`.

In [ ]:
# 1. Verify CUDA GPU availability
import torch
assert torch.cuda.is_available(), 'Please select a GPU runtime (Runtime -> Change runtime type -> T4 GPU)'
print('GPU Device:', torch.cuda.get_device_name(0))
print('VRAM Available (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

In [ ]:
# 2. Make Project Files Available & Locate PROJECT_ROOT
import os, subprocess
from pathlib import Path

# Check common locations or clone repository
candidates = [
    Path('/content/techzypher'),
    Path('/content/techzypher-sketch2shape-ai'),
    Path('/content'),
]
project_root = None
for c in candidates:
    if (c / 'ml').is_dir() and (c / 'worker.py').is_file():
        project_root = c
        break

if project_root is None:
    print('Cloning repository into /content/techzypher...')
    subprocess.run(['git', 'clone', 'https://github.com/arpit-thorat07/techzypher.git', '/content/techzypher'], check=True)
    project_root = Path('/content/techzypher')

assert (project_root / 'ml').is_dir(), f'ml directory not found in {project_root}'
assert (project_root / 'worker.py').is_file(), f'worker.py not found in {project_root}'
print(f'Verified project root containing ml/ at: {project_root}')

In [ ]:
# 3. Install uv package manager and Python 3.10
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.10'], check=True)
print('uv and Python 3.10 ready.')

In [ ]:
# 4. Create isolated virtual environments for ControlNet and TripoSR
import subprocess
from pathlib import Path
for name in ('controlnet-env', 'triposr-env'):
    python = Path(f'/content/{name}/bin/python')
    if not python.exists():
        subprocess.run(['uv', 'venv', f'/content/{name}', '--python', '3.10'], check=True)
    print(f'{name}:', subprocess.run([str(python), '--version'], capture_output=True, text=True, check=True).stdout.strip())

# Install PyTorch 2.1.2 (cu121) in both environments
for env in ['controlnet-env', 'triposr-env']:
    print(f'Installing PyTorch cu121 into {env}...')
    subprocess.run([
        'uv', 'pip', 'install', '--python', f'/content/{env}/bin/python',
        'torch==2.1.2', 'torchvision==0.16.2',
        '--index-url', 'https://download.pytorch.org/whl/cu121'
    ], check=True)
print('PyTorch installed successfully in both environments.')

In [ ]:
# 5. Install ControlNet dependencies
import subprocess
print('Installing ControlNet packages...')
subprocess.run([
    'uv', 'pip', 'install', '--python', '/content/controlnet-env/bin/python',
    'numpy==1.26.4', 'diffusers==0.27.2', 'transformers==4.38.2',
    'accelerate==0.29.3', 'huggingface-hub==0.25.2', 'safetensors', 'Pillow==10.2.0'
], check=True)
print('ControlNet environment ready.')

In [ ]:
# 6. Clone and patch TripoSR with marching-cubes fallback
import subprocess
from pathlib import Path

repo = Path('/content/TripoSR')
if not (repo / 'run.py').is_file():
    subprocess.run(['git', 'clone', 'https://github.com/VAST-AI-Research/TripoSR.git', str(repo)], check=True)

# Prepare sanitized requirements without gradio/torchmcubes
lines = (repo / 'requirements.txt').read_text().splitlines()
lines = [l for l in lines if 'torchmcubes' not in l and l.strip() != 'gradio' and not l.startswith('huggingface-hub')]
lines += ['huggingface-hub==0.17.3', 'numpy==1.26.4', 'scikit-image==0.22.0', 'onnxruntime-gpu']
Path('/content/triposr-req.txt').write_text('\n'.join(lines) + '\n')

print('Installing TripoSR packages...')
subprocess.run([
    'uv', 'pip', 'install', '--python', '/content/triposr-env/bin/python', '-r', '/content/triposr-req.txt'
], check=True)

# Apply skimage marching-cubes patch to isosurface.py
iso_path = repo / 'tsr/models/isosurface.py'
iso_src = iso_path.read_text()
old_import = 'from torchmcubes import marching_cubes'
mc_replacement = '''from skimage.measure import marching_cubes as _skimage_mc
import numpy as np
import torch

def marching_cubes(volume, level):
    verts, faces, _, _ = _skimage_mc(
        volume.detach().float().cpu().numpy(),
        level=level
    )
    verts = np.ascontiguousarray(verts[:, ::-1])
    faces = np.ascontiguousarray(faces.astype(np.int64))
    return torch.from_numpy(verts), torch.from_numpy(faces)
'''
if old_import in iso_src:
    iso_path.write_text(iso_src.replace(old_import, mc_replacement))
    print('TripoSR marching_cubes patched successfully.')
elif '_skimage_mc' in iso_src:
    print('TripoSR already patched.')

In [ ]:
# 7. Install Worker Server Dependencies in the main Colab environment
!pip install -q fastapi uvicorn python-multipart trimesh Pillow
print('Worker dependencies installed.')

In [ ]:
# 8. Configure Worker Authentication & Environment Paths
import os, secrets

# Retrieve from Colab Secrets or generate a cryptographically strong token
try:
    from google.colab import userdata
    token = userdata.get('ML_WORKER_TOKEN')
except Exception:
    token = None

if not token:
    token = secrets.token_urlsafe(32)

# Export environment variables (never print or log the plain token)
os.environ['ML_WORKER_TOKEN'] = token
os.environ['PROJECT_ROOT'] = str(project_root)
os.environ['CONTROLNET_PYTHON'] = '/content/controlnet-env/bin/python'
os.environ['TRIPOSR_PYTHON'] = '/content/triposr-env/bin/python'
os.environ['TRIPOSR_ROOT'] = '/content/TripoSR'

# Write configuration securely to /content/worker_config.env
with open('/content/worker_config.env', 'w') as f:
    f.write(f'ML_WORKER_TOKEN={token}\n')

print('Worker environment configured. Authentication token generated and stored securely in /content/worker_config.env.')

In [ ]:
# 9. Pre-flight Verification: Verify All Environments & Paths
from pathlib import Path

check_paths = [
    Path(os.environ['PROJECT_ROOT']) / 'ml',
    Path(os.environ['PROJECT_ROOT']) / 'worker.py',
    Path(os.environ['CONTROLNET_PYTHON']),
    Path(os.environ['TRIPOSR_PYTHON']),
    Path(os.environ['TRIPOSR_ROOT']) / 'run.py',
]
for p in check_paths:
    assert p.exists(), f'Missing prerequisite path: {p}'

assert os.environ.get('ML_WORKER_TOKEN'), 'ML_WORKER_TOKEN is not set'
print('All pre-flight checks passed! Worker environments and paths are ready.')

In [ ]:
# 10. Expose port 8000 via Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, time, re

tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stderr=subprocess.PIPE, text=True)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print('=' * 60)
print('PUBLIC WORKER TUNNEL URL:', public_url)
print('=' * 60)
print('\nCopy the URL above and token from /content/worker_config.env into your local backend/.env:')
print(f'ML_WORKER_URL={public_url}')
print('ML_WORKER_TOKEN=<token from /content/worker_config.env>')

In [ ]:
# 11. Launch Worker API Service
import os
os.chdir(os.environ['PROJECT_ROOT'])
!python worker.py